# chess-rl-bench — L3 training on Kaggle

Clones the portfolio repo, trains the L3 perceptron, evaluates it against a reference policy, and prints a summary.

**Two requirements:**
- **Internet enabled** (for the clone + `pip install`; set in the kernel metadata).
- The GitHub repo must be **public and already contain `bench/kaggle/`** — the clone step below checks this and aborts loudly if not, so a green run is trustworthy.

The repo is cloned to `/kaggle/temp/repo` (NOT `/kaggle/working/`), so it is *not* captured as a kernel output artifact — only `out/` is. Run the tiny validation config first, confirm it works, then scale up.

In [ ]:
# 1. Fetch the code and install dependencies.
#    Clone OUTSIDE /kaggle/working so the repo is not uploaded as a kernel output.
import os, shutil, subprocess

REPO_URL = 'https://github.com/cklt-90/Chess-dumbAI-benchmarking.git'
REPO = '/kaggle/temp/repo'

shutil.rmtree(REPO, ignore_errors=True)
os.makedirs(os.path.dirname(REPO), exist_ok=True)
subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO], check=True)

# Fail LOUDLY if the training script is missing -- this is exactly the failure
# that made a previous run look 'done' while doing nothing.
SCRIPT = os.path.join(REPO, 'bench', 'kaggle', 'kaggle_l3_train.py')
assert os.path.isfile(SCRIPT), (
    f'MISSING {SCRIPT} -- the GitHub repo does not contain bench/kaggle/. '
    'Was the commit pushed, and is the repo public?'
)
print('Found training script:', SCRIPT)

# Ensure deps (Kaggle usually has both; this is a cheap no-op if so).
subprocess.run(['pip', 'install', '-q', 'python-chess', 'numpy'], check=True)
print('Dependencies ready.')

## Run configuration

Edit the values below. Suggested **first run (validation)**: `MODE='both'`, `GAMES=20`, `POSITIONS=300`, `SEARCH_DEPTH=3`, `EVAL_GAMES=20` — confirms the clone, install, training, and eval all work on Kaggle.

**Scale run (measured rates, 2026-10-05):** on a Kaggle CPU, depth-3 labelling runs at ~**1.48 s/position** (vs ~0.24 s locally — the earlier estimate was ~6× optimistic). So **4k positions ≈ 1.6 h**, and 24k positions would be ~10 h. Recommended first scale run: `MODE='supervised'`, `POSITIONS=4000`.

**Do not scale the `selfplay`/`both` arms yet.** L3 self-play produces no usable outcome signal at this regime — short ply caps leave every game unfinished (weak shaped-only gradient) and long caps end 40/40 in `insufficient_material` draws (zero reward). See `bench/KAGGLE-PLAN.md`; fix and re-test locally first.

Labelling cost scales steeply with depth: ~0.25 s/pos (d2, est.), ~1.48 s/pos (d3, measured), ~27 s/pos (d4, est.). Keep depth 2–3 at scale; depth 4+ is not viable.

In [ ]:
MODE = 'both'
GAMES = 20
POSITIONS = 300
SEARCH_DEPTH = 3
EVAL_AGAINST = 'material'
EVAL_GAMES = 20
SEED = 7
# Scale run (supervised only, ~1.6h at measured ~1.48 s/pos depth-3):
#   MODE='supervised', POSITIONS=4000, SEARCH_DEPTH=3, EVAL_GAMES=40
# Self-play / both arms are NOT worth scaling until the signal finding is fixed.

OUT = '/kaggle/working/out'
cmd = (
    f'python {SCRIPT} '
    f'--mode {MODE} --games {GAMES} --positions {POSITIONS} '
    f'--search-depth {SEARCH_DEPTH} --eval-against {EVAL_AGAINST} '
    f'--eval-games {EVAL_GAMES} --seed {SEED} --out-dir {OUT}'
)
print('Running:', cmd)
subprocess.run(cmd, shell=True, check=True)

In [ ]:
# 3. Report. Assert the artifacts actually exist, so the cell cannot
#    'succeed' on a run that produced nothing.
import pathlib

out = pathlib.Path(OUT)
summary = out / 'summary.md'
assert summary.is_file(), f'No summary.md in {out} -- training did not complete.'

print(summary.read_text())
print('\nArtifacts in /kaggle/working/out (downloadable from the notebook output):')
for p in sorted(out.iterdir()):
    print(' -', p.name, f'({p.stat().st_size/1024:.0f} KB)')